# Task 1.1 — Core Contribution / Architecture
**Paper:** Incremental Learning of Temporally-Coherent Gaussian Mixture Models  
**Authors:** Ognjen Arandjelovic & Roberto Cipolla, Cambridge (BMVC / PAMI era)  
**Random Seed:** 42 (set at top of each notebook)


## Step-by-Step Method Description

**Step 1: GMM Parameterisation**
- Description: The data stream is modelled as a Gaussian Mixture Model with M components. Each component i has a prior α_i, mean μ_i, and full covariance C_i.
- Reference: Equations (1) and (2), Section 2
- Purpose: Establishes the generative model whose parameters will be updated incrementally.

**Step 2: Fixed-Complexity Update (Eq. 7 & 8)**
- Description: When a new observation x arrives, its soft responsibility p(i|x) is computed for each component. The parameters (α_i, μ_i, C_i) are then updated using a closed-form formula that only requires the current model parameters and the scalar E_i (running sum of responsibilities). No stored historical data is needed.
- Reference: Equations (3)–(8), Section 2.3
- Purpose: Allows the model to absorb a new point in O(M·D²) time without keeping raw data in memory. This is the core memory-efficiency contribution.

**Step 3: Model Splitting via Historical GMM (Eq. 9 & 10)**
- Description: The paper stores the oldest model of the same complexity — the Historical GMM G^(h). By computing the "difference component" between the historical and current GMM (via Eq. 9–10), the method postulates a new Gaussian component to capture drift/new structure not present at the historical snapshot.
- Reference: Section 2.4, Equations (9) and (10), Algorithm 1 Step 2
- Purpose: The Historical GMM is what separates this method from naive one-point-at-a-time EM. Without it, a single new point never carries enough evidence to justify splitting a component.

**Step 4: Component Merging via MDL (Eq. 11–16)**
- Description: After tentative splitting, every pair of components is evaluated for merging. The expected description length (EDL) is computed for the split and merged configurations using the Bhattacharyya distance between Gaussian pairs (Eq. 14). If merging yields a lower EDL, the components are merged.
- Reference: Section 2.5, Equations (11)–(16), Algorithm 1 Steps 3–6
- Purpose: Controls model complexity in an principled way. MDL prevents over-splitting by penalising models with too many free parameters.

**Step 5: Historical GMM Update**
- Description: When a permanent increase in component count occurs (after a split survives MDL pruning), the Historical GMM is replaced by the current model, resetting the reference for future splits.
- Reference: Section 2.4, paragraph 2; Algorithm 1 Step 2
- Purpose: Ensures the Historical GMM always tracks the most recent point of model order stability.

---
## Final Summary
This paper solves the problem of learning a Gaussian Mixture Model incrementally when data arrives one point at a time, memory is constrained, and the number of clusters is unknown; the key innovation is the Historical GMM — a stored snapshot of the oldest model of the same complexity — which provides sufficient evidence to detect when a new cluster should be created, something that a single new data point alone can never supply.
